In [1]:
# import the necessary libraries

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image

In [2]:
# import the dataset
dp = 'PetImages'

In [3]:
# image preprocessing
datagen = ImageDataGenerator(
    # normalize pixels
    rescale = 1./255,

    # data augmentation
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,

    # split dataset
    validation_split = 0.2 # validation split used to create train and validation data
)

In [4]:
# load training data

# training data
train_data = datagen.flow_from_directory(
    dp,
    target_size = (128,128),
    batch_size = 32,
    class_mode = 'binary',
    subset = 'training'
)

Found 20000 images belonging to 2 classes.


In [5]:
# load validation data
val_data = datagen.flow_from_directory(
    dp,
    target_size = (128,128),
    batch_size = 32,
    class_mode = 'binary',
    subset = 'validation'
)

Found 4998 images belonging to 2 classes.


In [6]:
print(train_data.class_indices)

{'Cat': 0, 'Dog': 1}


In [7]:
# building CNN model 
model = keras.Sequential()

In [8]:
# first convolution layer
model.add(
    keras.layers.Conv2D(
        filters = 32,
        kernel_size = (3,3),
        activation = 'relu',
        input_shape = (128,128,3)
    )
)

/Users/santhoshib/miniconda3/envs/interviewai/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# first pooling layer
model.add(
    keras.layers.MaxPooling2D(
        pool_size = (2,2)
    )
)

In [10]:
# second convolution layer
model.add(
    keras.layers.Conv2D(
        filters = 32,
        kernel_size = (3,3),
        activation = 'relu'
    )
)

In [11]:

# second pooling layer 
model.add(
    keras.layers.MaxPooling2D(
        pool_size = (2,2)
    )
)

In [12]:
# flatten
model.add(keras.layers.Flatten())

In [13]:
# dense layer 
model.add(
    keras.layers.Dense(
        units = 128,
        activation = 'relu'
    )
)

In [14]:
# output layer
model.add(
    keras.layers.Dense(
        units = 1,
        activation = 'sigmoid'
    )
)

In [15]:
# model architecture
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,686,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,696,801 (14.10 MB)

 Trainable params: 3,696,801 (14.10 MB)

 Non-trainable params: 0 (0.00 B)

## 1st convolutional layer
#### (None,126,126,Channels)
#### (Batch size, height, weight, no.of filters)
- input = 128 * 128 * 3
- filter = 3*3
- No.of filters = 32
#### output size formula
- output = ((input-kernel)/(stride))+1
- stride = 1
- output = ((128-3)/1)+1 
         = 126

- height = 126
- width = 126
- channels ---> 32
- final output ---> (None,126,126,32)

#### kernel 
- Formula = (kh * kw * InputChannels + 1) * Filters
- kernel = 3*3
- InputChannels = 3
- filters = 32
- substitute in formula 
    = (3 * 3 * 3 + 1) * 32
    = 896

## 1st Pooling2D
- MaxPooling2D((2,2))
- pooling = No learning, No weights
(only reduces size)
- input = 126 * 126 * 32
- pool = 2*2
- Formula = 126/2
          = 63
- output = 63 * 63 * 32
- Final = (None, 63, 63, 32)

## 2nd conv2D
- input from pooling = (None, 63 * 63 * 32)
- kernel = 3*3
- output = (63-3)+1
         = 61
- (None, 61, 61, 32)
- params = (3 * 3 * 32 + 1) * 32
        = 9248


## 2nd Pooling2D
- (None, 30, 30, 32)
- param = 0
- input = (61, 61, 32)
- pooling = 2*2
- floor division = 61 // 2 
                 = 30
- final = (None, 30, 30, 32)
- no parameters



## Flatten 
- input = 30 * 30 * 32
- output = 30 * 30 * 32 = 28800 (numbers)
- flatten = it will convert 30 ---> 10 vector... (Flatten only reshapes...)



## Dense layer
- input = 28800
- Neurons = 128
- parameters = (input * neurons) + bias
             = (28800 * 128) + 128
             = 3686528

## Output layer
- input = 128
- output = 1
- output shape = (None, 1)

- meaning = single prediction
- --> cat or dog
- parameter calculation = 128 * 1 + 1
                        = 129
#### Input --> Conv1 --> Pool1 --> Conv2 --> Pool2 --> Flatten --> Dense --> Output

In [16]:
print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [17]:
# compile cnn
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [18]:
history = model.fit(
    train_data,
    epochs=10,
    validation_data = val_data
)

ImportError: This requires the scipy module. You can install it via `pip install scipy`

In [ ]:
# evaluate the model
loss, accuracy = model.evaluate(val_data)
print("Accuracy : ",accuracy)

157/157 ━━━━━━━━━━━━━━━━━━━━ 18s 113ms/step - accuracy: 0.8055 - loss: 0.4343
Accuracy :  0.8055222034454346
